# Supplemental PBPK Scenario Evaluation (Table S2)

This notebook reproduces the supplemental PBPK scenario evaluation reported in Table S2. It evaluates all concentration-time simulation folders under `simulation/TableS2/` against the observed concentration-time data in `observed/`, then exports per-scenario and cross-scenario performance metrics under `evaluation/TableS2/`.

The notebook is intended as a reproducibility record for the supplemental PBPK analysis. The key outputs are analysis-ready CSV files; figures and manuscript-level interpretation are handled in the main comparison notebook where needed.


In [1]:
import os
import numpy as np
import pandas as pd
from scipy.interpolate import interp1d
from scipy.interpolate import PchipInterpolator
pd.set_option("display.float_format", lambda x: f"{x:.4f}")


SIM_BASE = "./simulation/TableS2"
EVAL_BASE = "./evaluation"
OBS_PATH = "./observed/observed_data_cleaned_deduplicated.csv"
eval_running = "TableS2"


# calculate evaluation metrics and summary statistics

In [2]:

# ============================================================
# Helper: AUC with linear-up/log-down rule
# ============================================================
def calc_auc(time, conc):
    auc = 0.0
    for i in range(len(time) - 1):
        t1, t2 = time[i], time[i+1]
        c1, c2 = conc[i], conc[i+1]

        if c2 > c1:  # linear up
            auc += (c1 + c2) / 2 * (t2 - t1)
        else:        # log down
            if c1 > 0 and c2 > 0 and c1 != c2:
                auc += (c1 - c2) / np.log(c1 / c2) * (t2 - t1)
            else:
                auc += (c1 + c2) / 2 * (t2 - t1)
    return auc


# ============================================================
# Helper: Evaluate one compound
# folds: tuple/list of fold thresholds for coverage metrics
# ============================================================
def evaluate_one_compound(obs_df_c, sim_df_c, folds=(2, 3, 5, 10)):

    cid = obs_df_c["compound_id"].iloc[0]

    # ---------- interpolate simulated concentrations to observed sampling times ----------
    f = PchipInterpolator(sim_df_c["Time_hr"], sim_df_c["mean"])
    pred = f(obs_df_c["Time_hr"].values)

    obs = obs_df_c["Conc_mgl"].values
    time = obs_df_c["Time_hr"].values

    # Remove non-positive
    mask = (pred > 0) & (obs > 0)
    pred = pred[mask]
    obs = obs[mask]
    time = time[mask]

    # ---------- point-wise log2 errors ----------
    log2_ratio = np.log2(pred / obs)
    rel_log2 = np.mean(log2_ratio)
    abs_log2 = np.mean(np.abs(log2_ratio))

    # ---------- Cmax ----------
    cmax_pred = np.max(pred)
    cmax_obs  = np.max(obs)
    fe_cmax_rel = np.log2(cmax_pred / cmax_obs)     # direction (bias)
    fe_cmax_abs = np.abs(fe_cmax_rel)               # magnitude

    # ---------- AUC ----------
    auc_pred = calc_auc(time, pred)
    auc_obs  = calc_auc(time, obs)
    fe_auc_rel = np.log2(auc_pred / auc_obs)
    fe_auc_abs = np.abs(fe_auc_rel)

    # ---------- Tmax difference ----------
    tmax_pred = time[np.argmax(pred)]
    tmax_obs  = time[np.argmax(obs)]
    delta_tmax = tmax_pred - tmax_obs

    # ---------- Within X-fold coverage metrics ----------
    # Use absolute log2 error to determine whether values are within each fold threshold
    coverage = {}
    abs_log2_point = np.abs(log2_ratio)

    for fold in folds:
        thr = np.log2(fold)

        # 1) Time-point coverage: fraction of observed time points within each fold threshold
        pct_time_within = np.mean(abs_log2_point <= thr)  # range 0-1
        coverage[f"time_within_{fold}fold"] = pct_time_within

        # 2) Cmax coverage：compound-level 0/1
        coverage[f"cmax_within_{fold}fold"] = float(fe_cmax_abs <= thr)

        # 3) AUC coverage：compound-level 0/1
        coverage[f"auc_within_{fold}fold"] = float(fe_auc_abs <= thr)

    # ---------- assemble result ----------
    result = {
        "compound_id": cid,
        "rel_log2": rel_log2,
        "abs_log2": abs_log2,
        "fe_cmax_rel": fe_cmax_rel,
        "fe_cmax_abs": fe_cmax_abs,
        "fe_auc_rel": fe_auc_rel,
        "fe_auc_abs": fe_auc_abs,
        "delta_tmax": delta_tmax,
    }

    # Add coverage fields to the result dictionary
    result.update(coverage)
    return result


“PCHIP preserves monotonicity and the shape of the data, avoiding overshoot.
It is recommended for scientific datasets where spline oscillation is undesirable.”

In [3]:
# ============================================================
# Main Pipeline
# ============================================================

# ------------------ Load Observed ---------------------------
obs_df = pd.read_csv(OBS_PATH)
obs_df["Conc_mgl"] = obs_df["Conc_ng_ml"] / 1000
obs_df = obs_df[obs_df["Conc_mgl"] > 0]
obs_df = obs_df.sort_values(["compound_id", "Time_hr"])

# ============================================================
# Loop through simulation folders
# ============================================================

all_results = [] 

for scenario in os.listdir(SIM_BASE):
    scenario_path = os.path.join(SIM_BASE, scenario)
    ct_file = os.path.join(scenario_path, "ct_wide_all.xlsx")

    if not os.path.isfile(ct_file):
        continue  # skip folders without ct_wide_all.xlsx

    print(f"Processing scenario: {scenario}")

    sim_df = pd.read_excel(ct_file)
    sim_df = sim_df[sim_df["mean"] > 0]
    sim_df = sim_df.sort_values(["compound_id", "Time_hr"])

    # Filter observed to only compounds present in simulation
    obs_use = obs_df[obs_df["compound_id"].isin(sim_df["compound_id"].unique())]

    # Evaluate
    results = []
    for cid, obs_df_c in obs_use.groupby("compound_id"):
        sim_df_c = sim_df[sim_df["compound_id"] == cid]
        res = evaluate_one_compound(obs_df_c, 
                                    sim_df_c,
                                    folds=(2,3,5,10))
        results.append(res)

    results_df = pd.DataFrame(results)

    # Save output
    out_dir = os.path.join(EVAL_BASE, eval_running)
    os.makedirs(out_dir, exist_ok=True)
    out_file = os.path.join(out_dir, f"{scenario}_evaluation_summary.csv")
    results_df.to_csv(out_file, index=False)

    print(f"Finished: {scenario} → {out_file}")

    # Add scenario name to results
    results_df["scenario"] = scenario
    
    
    # Store for overall summary later
    all_results.append(results_df)
    
    
print("All scenarios processed.")


Processing scenario: v4_ML
Finished: v4_ML → ./evaluation/TableS2/v4_ML_evaluation_summary.csv
Processing scenario: v0_run0
Finished: v0_run0 → ./evaluation/TableS2/v0_run0_evaluation_summary.csv
Processing scenario: h2_run2
Finished: h2_run2 → ./evaluation/TableS2/h2_run2_evaluation_summary.csv
Processing scenario: v1_run0
Finished: v1_run0 → ./evaluation/TableS2/v1_run0_evaluation_summary.csv
Processing scenario: h2_run3
Finished: h2_run3 → ./evaluation/TableS2/h2_run3_evaluation_summary.csv
Processing scenario: s1_run1
Finished: s1_run1 → ./evaluation/TableS2/s1_run1_evaluation_summary.csv
Processing scenario: h1_run1
Finished: h1_run1 → ./evaluation/TableS2/h1_run1_evaluation_summary.csv
Processing scenario: v3_CLsys
Finished: v3_CLsys → ./evaluation/TableS2/v3_CLsys_evaluation_summary.csv
Processing scenario: h2_run1
Finished: h2_run1 → ./evaluation/TableS2/h2_run1_evaluation_summary.csv
Processing scenario: v1_run4
Finished: v1_run4 → ./evaluation/TableS2/v1_run4_evaluation_summa

In [4]:
# ============================================================
# Overall Summary Across Scenarios
# ============================================================

if len(all_results) > 0:
    overall_df = pd.concat(all_results, ignore_index=True)

    # ----- 1. Summary by scenario (mean/median/std + coverage) -----
    scenario_summary = (
        overall_df
        .groupby("scenario")
        .agg({
            # curve-level
            "rel_log2": ["mean", "median", "std"],
            "abs_log2": ["mean", "median", "std"],
            # Cmax / AUC FE
            "fe_cmax_rel": ["mean", "median", "std"],
            "fe_cmax_abs": ["mean", "median", "std"],
            "fe_auc_rel":  ["mean", "median", "std"],
            "fe_auc_abs":  ["mean", "median", "std"],
            "delta_tmax":  ["mean", "median", "std"],
            # coverage (all are proportions 0–1)
            "time_within_2fold": ["mean"],
            "time_within_3fold": ["mean"],
            "time_within_5fold": ["mean"],
            "time_within_10fold": ["mean"],
            "cmax_within_2fold": ["mean"],
            "cmax_within_3fold": ["mean"],
            "cmax_within_5fold": ["mean"],
            "cmax_within_10fold": ["mean"],
            "auc_within_2fold":  ["mean"],
            "auc_within_3fold":  ["mean"],
            "auc_within_5fold":  ["mean"],
            "auc_within_10fold": ["mean"],
        })
    )

    scenario_summary.columns = [
        "_".join(col).rstrip("_") for col in scenario_summary.columns
    ]
    scenario_summary = scenario_summary.reset_index()

    # ----- 2. Summary by metric (boxplot-ready format) -----
    long_format = overall_df.melt(
        id_vars=["compound_id", "scenario"],
        value_vars=[
            "rel_log2", "abs_log2",
            "fe_cmax_rel", "fe_cmax_abs",
            "fe_auc_rel",  "fe_auc_abs"
        ],
        var_name="metric",
        value_name="value"
    )

    # ----- Create output folder -----
    overall_out = os.path.join(EVAL_BASE, eval_running, "overall")
    os.makedirs(overall_out, exist_ok=True)

    # ----- Save files with 4 decimal places -----
    scenario_summary.to_csv(
        os.path.join(overall_out, "evaluation_summary_by_scenario.csv"),
        index=False, float_format="%.4f"
    )
    long_format.to_csv(
        os.path.join(overall_out, "evaluation_long_format.csv"),
        index=False, float_format="%.4f"
    )
    overall_df.to_csv(
        os.path.join(overall_out, "evaluation_all_raw.csv"),
        index=False, float_format="%.4f"
    )

    print("\n===== Overall Summary Saved =====")
    print(f"Scenario summary → {overall_out}/evaluation_summary_by_scenario.csv")
    print(f"Long-format data → {overall_out}/evaluation_long_format.csv")
    print(f"Raw combined data → {overall_out}/evaluation_all_raw.csv")

   

else:
    print("No scenario results collected.")



===== Overall Summary Saved =====
Scenario summary → ./evaluation/TableS2/overall/evaluation_summary_by_scenario.csv
Long-format data → ./evaluation/TableS2/overall/evaluation_long_format.csv
Raw combined data → ./evaluation/TableS2/overall/evaluation_all_raw.csv


## Output Index

Primary Table S2 outputs generated by this notebook:

- `evaluation/TableS2/overall/evaluation_all_raw.csv`: compound-level raw evaluation results for every scenario.
- `evaluation/TableS2/overall/evaluation_long_format.csv`: long-format metric table for downstream plotting or auditing.
- `evaluation/TableS2/overall/evaluation_summary_by_scenario.csv`: scenario-level summary table containing the main Table S2 metric values.
- `evaluation/TableS2/overall/scenario_summary_key_ranking.csv`: rank-based summary using concentration-time profile error, Cmax/AUC error, and within-2-fold coverage.
- `evaluation/TableS2/overall/scenario_summary_key_rel.csv`: directional-bias summary using relative log2 errors.
- `evaluation/TableS2/*_evaluation_summary.csv`: per-scenario metric summaries.

Reported metrics include concentration-time profile relative and absolute log2 errors, Cmax and AUC fold-errors, Tmax differences, and time-point/Cmax/AUC coverage within predefined fold-error thresholds.


In [5]:
overall_df.scenario.unique()

array(['v4_ML', 'v0_run0', 'h2_run2', 'v1_run0', 'h2_run3', 's1_run1',
       'h1_run1', 'v3_CLsys', 'h2_run1', 'v1_run4', 's1_run3', 'v2_CLint',
       'h1_run3', 'h1_run2', 's1_run2'], dtype=object)

In [6]:
scenario_summary.columns

Index(['scenario', 'rel_log2_mean', 'rel_log2_median', 'rel_log2_std',
       'abs_log2_mean', 'abs_log2_median', 'abs_log2_std', 'fe_cmax_rel_mean',
       'fe_cmax_rel_median', 'fe_cmax_rel_std', 'fe_cmax_abs_mean',
       'fe_cmax_abs_median', 'fe_cmax_abs_std', 'fe_auc_rel_mean',
       'fe_auc_rel_median', 'fe_auc_rel_std', 'fe_auc_abs_mean',
       'fe_auc_abs_median', 'fe_auc_abs_std', 'delta_tmax_mean',
       'delta_tmax_median', 'delta_tmax_std', 'time_within_2fold_mean',
       'time_within_3fold_mean', 'time_within_5fold_mean',
       'time_within_10fold_mean', 'cmax_within_2fold_mean',
       'cmax_within_3fold_mean', 'cmax_within_5fold_mean',
       'cmax_within_10fold_mean', 'auc_within_2fold_mean',
       'auc_within_3fold_mean', 'auc_within_5fold_mean',
       'auc_within_10fold_mean'],
      dtype='object')

>“We ranked parameterisation strategies primarily by the median Absolute Log2 Error across compounds, in line with Geci et al. (2024). Strategies with low median Absolute Log2 Error, low bias (median Relative Log2 Error close to zero), and small Cmax and AUC fold-errors were considered preferable. Additionally, we examined the proportion of compounds predicted within 2-fold for Cmax and AUC as a global coverage measure.”

>“We separated directional bias (relative log₂ fold errors) from error magnitude (absolute log₂ fold errors) consistently for both full concentration–time profiles and key PK metrics (Cmax and AUC), to avoid cancellation effects and enable robust ranking of simulation scenarios.”

In [7]:
 # quick view of key metrics
scenario_summary[
        ["scenario",
            "abs_log2_median",
            "fe_cmax_abs_median",
            "fe_auc_abs_median",
            "time_within_2fold_mean",
            "cmax_within_2fold_mean",
            "auc_within_2fold_mean"]
    ]


,scenario,abs_log2_median,fe_cmax_abs_median,fe_auc_abs_median,time_within_2fold_mean,cmax_within_2fold_mean,auc_within_2fold_mean
0,h1_run1,1.0101,1.0395,0.4361,0.5932,0.4634,0.8049
1,h1_run2,1.0851,0.9132,0.5039,0.5762,0.5854,0.8537
2,h1_run3,1.0405,0.8773,0.4803,0.5752,0.6098,0.8537
3,h2_run1,1.7149,1.2165,0.9594,0.3586,0.4146,0.5366
4,h2_run2,1.2632,0.9939,0.9207,0.4489,0.5122,0.5366
5,h2_run3,1.3213,0.9883,0.8908,0.4414,0.5122,0.5610
6,s1_run1,0.6754,0.4453,0.3628,0.7685,0.7073,0.8537
7,s1_run2,0.7415,0.4903,0.4080,0.7459,0.7561,0.8780
8,s1_run3,0.7415,0.4965,0.4080,0.7459,0.7561,0.8780
9,v0_run0,0.6373,0.4551,0.2831,0.7670,0.7561,0.8780


In [8]:
# ============================================================
# Scenario ranking
# ============================================================

sc = scenario_summary.copy()

# ---- 1. Generate ranks for each metric ----
# Error metrics: smaller is better -> ascending=True
sc["rk_abslog2"]   = sc["abs_log2_median"].rank(method="average", ascending=True)
sc["rk_fe_auc"]    = sc["fe_auc_abs_median"].rank(method="average", ascending=True)
sc["rk_fe_cmax"]   = sc["fe_cmax_abs_median"].rank(method="average", ascending=True)

# Coverage metrics: larger is better -> ascending=False
sc["rk_time2"]     = sc["time_within_2fold_mean"].rank(method="average", ascending=False)
sc["rk_auc2"]      = sc["auc_within_2fold_mean"].rank(method="average", ascending=False)
sc["rk_cmax2"]     = sc["cmax_within_2fold_mean"].rank(method="average", ascending=False)

# ---- 2. Set metric weights (all equal by default) ----
w_abslog2 = 1.0   # overall concentration-time profile error
w_auc      = 1.0   # AUC error
w_cmax     = 1.0   # Cmax error
w_time2    = 1.0   # time-point 2-fold coverage
w_auc2     = 1.0   # AUC 2-fold coverage
w_cmax2    = 1.0   # Cmax 2-fold coverage

# Composite rank score, similar to a Borda count; smaller is better
sc["rank_score"] = (
    w_abslog2 * sc["rk_abslog2"] +
    w_auc     * sc["rk_fe_auc"]   +
    w_cmax    * sc["rk_fe_cmax"]  +
    w_time2   * sc["rk_time2"]    +
    w_auc2    * sc["rk_auc2"]     +
    w_cmax2   * sc["rk_cmax2"]
)

sc = sc.sort_values("rank_score").reset_index(drop=True)

print("\n=== Scenario ranking (lower score is better) ===")
sc[["scenario", "rank_score", "abs_log2_median", "fe_auc_abs_median", "fe_cmax_abs_median", "time_within_2fold_mean", "auc_within_2fold_mean", "cmax_within_2fold_mean"]]


=== Scenario ranking (lower score is better) ===


,scenario,rank_score,abs_log2_median,fe_auc_abs_median,fe_cmax_abs_median,time_within_2fold_mean,auc_within_2fold_mean,cmax_within_2fold_mean
0,v0_run0,13.5000,0.6373,0.2831,0.4551,0.7670,0.8780,0.7561
1,v1_run4,14.5000,0.5710,0.2942,0.3895,0.7882,0.8780,0.6829
2,s1_run1,20.5000,0.6754,0.3628,0.4453,0.7685,0.8537,0.7073
3,s1_run2,24.0000,0.7415,0.4080,0.4903,0.7459,0.8780,0.7561
4,s1_run3,28.0000,0.7415,0.4080,0.4965,0.7459,0.8780,0.7561
5,v1_run0,33.0000,0.7467,0.3709,0.4949,0.7492,0.8049,0.7073
6,v2_CLint,45.0000,0.8925,0.6208,0.4679,0.6592,0.7561,0.6829
7,h1_run3,53.0000,1.0405,0.4803,0.8773,0.5752,0.8537,0.6098
8,h1_run2,57.0000,1.0851,0.5039,0.9132,0.5762,0.8537,0.5854
9,h1_run1,59.5000,1.0101,0.4361,1.0395,0.5932,0.8049,0.4634


In [9]:
sc.to_csv(os.path.join(overall_out, "scenario_summary_key_ranking.csv"),
          index=False, float_format="%.3f")

In [10]:
sc_key = sc[['scenario','rel_log2_mean','rel_log2_median',
     'fe_cmax_rel_mean','fe_cmax_rel_median', 
     'fe_auc_rel_mean', 'fe_auc_rel_median']]

sc_key.to_csv(os.path.join(overall_out, "scenario_summary_key_rel.csv"),
          index=False, float_format="%.3f")
     

In [11]:
sc_key

,scenario,rel_log2_mean,rel_log2_median,fe_cmax_rel_mean,fe_cmax_rel_median,fe_auc_rel_mean,fe_auc_rel_median
0,v0_run0,-0.0282,0.1120,-0.3519,-0.2838,0.0677,0.0960
1,v1_run4,0.1936,0.3380,-0.5515,-0.3054,0.2128,0.2226
2,s1_run1,0.2162,0.3447,-0.6978,-0.3964,0.2833,0.2935
3,s1_run2,0.0925,0.1664,-0.6657,-0.4581,0.2939,0.2968
4,s1_run3,0.0770,0.1478,-0.6915,-0.4734,0.2832,0.2860
5,v1_run0,0.2367,0.3514,-0.6728,-0.3966,0.3028,0.2942
6,v2_CLint,0.0198,0.2797,-0.6913,-0.4679,0.2165,0.2180
7,h1_run3,-0.8848,-0.0142,0.0092,-0.1624,-0.0529,0.1091
8,h1_run2,-0.8244,-0.0425,0.0542,-0.1683,0.0004,0.1166
9,h1_run1,-0.3237,-0.0156,-0.1288,0.0081,0.0029,0.1500
